# Superstore Retail Analytics - EDA and Answering Questions

**Continuing from:** `feature_engineering_analysis.ipynb` — this notebook picks up once the data is *prepared*, and turns it into something that can actually answer business questions.

**Workflow:**

This is the same repeatable process regardless of the dataset — the checklist we'll apply to the Superstore data in Part 2 below.

1. **Exploratory Data Analysis** — univariate → bivariate → multivariate, in that order.
2. **Statistical Analysis** — Correlation, Distribution & Trend
2. **Answering the Questions** — go back to Step 1 and directly resolve each question with a specific table or aggregation.
3. **Insight Synthesis** — summarize findings in plain language, including caveats inherited from cleaning decisions.
4. **Handoff Prep** — name and preserve the tables/features the next stage (visualization) will need.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow
import squarify
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [3]:
# Reading the clean data and make a copy of the original file to be the single source of truth
df_prepared = pd.read_parquet("Sample-Superstore2019_prepared.parquet", engine="pyarrow")
df_prepared.shape

(9986, 31)

In [4]:
rfm = pd.read_parquet("Sample-Superstore2019_rfm.parquet", engine="pyarrow")
rfm.shape

(793, 11)

### Step 1 — Exploratory Data Analysis

Univariate first (what does one feature look like on its own), then bivariate (how does it relate to profit), then multivariate (multiple dimensions at once).

In [48]:
customer_features = (
    df_prepared.groupby(["Customer ID", "Customer Name"], observed=True)
      .agg(
          
          total_sales=("Sales", "sum"),
          total_profit=("Profit", "sum"),
          total_quantity=("Quantity", "sum"),
          order_count=("Order ID", "nunique"),
          avg_order_value=("Sales", "mean"),
      )
      .reset_index()
)

In [5]:
# Univariate: distribution of the new per-row features
print(df_prepared["Discount Bucket"].value_counts())
print()
print(df_prepared["Shipping Days"].describe())
print()
print(df_prepared["Profit Margin"].describe())

Discount Bucket
None         4793
Low          3801
High          856
Medium        536
Very High       0
Name: count, dtype: int64

count    9986.000000
mean        3.958442
std         1.748245
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: Shipping Days, dtype: float64

count    9986.000000
mean       12.018479
std        46.689386
min      -275.000000
25%         7.500000
50%        27.000000
75%        36.250000
max        50.000000
Name: Profit Margin, dtype: float64


In [6]:
# Bivariate: Profit against Discount Bucket, Sales against calendar features
print(df_prepared.groupby("Discount Bucket", observed=True)["Profit"].mean())
print()
print(df_prepared.groupby("Order Month", observed=True)["Sales"].sum())

Discount Bucket
None       66.901973
Low        26.497411
Medium   -109.710720
High      -89.438144
Name: Profit, dtype: float64

Order Month
1      94924.8356
2      59751.2514
3     204959.8088
4     137188.7966
5     155028.8117
6     152718.6793
7     147148.0370
8     159044.0630
9     307600.8257
10    200322.9847
11    351916.6910
12    324904.7875
Name: Sales, dtype: float64


In [7]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df_prepared.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

,Sales,Profit
Region,,
West,725355.4885,108404.3777
East,677906.3680,91353.8822
South,391007.8250,46549.1972
Central,501239.8908,39706.3625


In [8]:
# Multiple aggregations at once with .agg()
category_summary = df_prepared.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

,total_sales,avg_profit,orders
Category,,,
Furniture,741432.0433,8.674036,2119
Office Supplies,718317.7920,20.300133,6022
Technology,835759.7370,78.800073,1845


In [9]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df_prepared,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

Category,Furniture,Office Supplies,Technology
Region,,,
Central,340.534644,117.458801,405.753124
East,346.683053,119.837751,495.278469
South,353.511492,126.400376,508.492973
West,357.302325,116.422377,421.219893


In [10]:
# Boolean indexing: orders sold at a loss
loss_orders = df_prepared[df_prepared["Profit"] < 0]
print("Loss-making orders:", len(loss_orders))
loss_orders[["Order ID", "Product Name", "Sales", "Profit"]]

Loss-making orders: 1870


,Order ID,Product Name,Sales,Profit
3,US-2017-108966,Bretford CR4500 Series Slim Rectangular Table,957.5775,-383.0310
14,US-2017-118983,Holmes Replacement Filter for HEPA Air Cleaner...,68.8100,-123.8580
15,US-2017-118983,Storex DuraTech Recycled Plastic Frosted Binders,2.5440,-3.8160
23,US-2019-156909,"Global Deluxe Stacking Chair, Gray",71.3720,-1.0196
27,US-2017-150630,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,-1665.0522
...,...,...,...,...
9912,CA-2018-149272,"GBC Pre-Punched Binding Paper, Plastic, White,...",22.3860,-35.8176
9913,CA-2016-111360,Acco Expandable Hanging Binders,5.7420,-4.5936
9923,CA-2017-104948,O'Sullivan Living Dimensions 3-Shelf Bookcases,683.3320,-40.1960
9929,CA-2018-164889,Hon 61000 Series Interactive Training Tables,71.0880,-1.7772


In [11]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df_prepared.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

,Sales,Profit
Region,,
West,725355.4885,108404.3777
East,677906.3680,91353.8822
South,391007.8250,46549.1972
Central,501239.8908,39706.3625


In [12]:
# Multiple aggregations at once with .agg()
category_summary = df_prepared.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

,total_sales,avg_profit,orders
Category,,,
Furniture,741432.0433,8.674036,2119
Office Supplies,718317.7920,20.300133,6022
Technology,835759.7370,78.800073,1845


In [13]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df_prepared,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

Category,Furniture,Office Supplies,Technology
Region,,,
Central,340.534644,117.458801,405.753124
East,346.683053,119.837751,495.278469
South,353.511492,126.400376,508.492973
West,357.302325,116.422377,421.219893


### Step 2 — Statistical Analysis


In [15]:
#create a Correlation matrix to see the correlation between the numerical features
numeric_features = df_prepared[["Sales", "Quantity", "Discount", "Profit", "Shipping Days", "Profit Margin"]]
correlation_matrix = numeric_features.corr()
correlation_matrix

,Sales,Quantity,Discount,Profit,Shipping Days,Profit Margin
Sales,1.000000,0.200927,-0.028216,0.479056,-0.007368,0.003485
Quantity,0.200927,1.000000,0.008599,0.066288,0.018318,-0.005320
Discount,-0.028216,0.008599,1.000000,-0.219459,0.000229,-0.864446
Profit,0.479056,0.066288,-0.219459,1.000000,-0.004621,0.223723
Shipping Days,-0.007368,0.018318,0.000229,-0.004621,1.000000,-0.011676
Profit Margin,0.003485,-0.005320,-0.864446,0.223723,-0.011676,1.000000


### Step 3 — Answering the Questions

Back to the five questions from Step 1 in Feature engineering— each one resolved directly with the features built above.

## Q1 — Which State generates the most total Profit, and which generates the biggest total loss?


In [16]:
state_profit = (
    df_prepared.groupby("State", observed=True)["Profit"]
      .sum()
      .sort_values(ascending=False)
)

print("Top 10 states by total Profit:")
display(state_profit.head(10).to_frame("Total Profit"))

print("\nBottom 10 states by total Profit (largest losses first):")
display(state_profit.tail(10).sort_values().to_frame("Total Profit"))

best_state = state_profit.idxmax()
worst_state = state_profit.idxmin()
print(f"\nAnswer: {best_state} generates the highest total Profit (${state_profit.max():,.2f}).")
print(f"The biggest total loss is generated by {worst_state} (${state_profit.min():,.2f}).")

Top 10 states by total Profit:


,Total Profit
State,
California,76381.3871
New York,74015.4622
Washington,33402.6517
Michigan,24463.1876
Virginia,18514.9002
Indiana,18382.9363
Georgia,16250.0433
Kentucky,11158.2690
Minnesota,10823.1874



Bottom 10 states by total Profit (largest losses first):


,Total Profit
State,
Texas,-25729.3563
Ohio,-16959.3178
Pennsylvania,-15559.9603
Illinois,-12607.8870
North Carolina,-7545.6547
Colorado,-6541.9291
Tennessee,-5341.6936
Arizona,-3427.9246
Florida,-3399.3017



Answer: California generates the highest total Profit ($76,381.39).
The biggest total loss is generated by Texas ($-25,729.36).


## Q2 — Is there a relationship between Discount and Profit?

In [ ]:
# Q2: relationship between Discount and Profit, using the bucket built in Step 2
discount_profit = df_prepared.groupby("Discount Bucket", observed=True)[["Profit margin" , "Profit per unit"]].mean()
discount_profit

Discount Bucket
None       66.901973
Low        26.497411
Medium   -109.710720
High      -89.438144
Name: Profit, dtype: float64

Answer: Discount has a negative linear relationship with Profit (Pearson r = -0.028215522858040774).
A negative value means that higher discounts are associated with lower profitability in this dataset.

## Q3 — Does Ship Mode relate to profitability or shipping time?

In [19]:
ship_mode_summary = (
    df_prepared.groupby("Ship Mode", observed=True)
      .agg(
          orders=("Order ID", "nunique"),
          avg_profit=("Profit", "mean"),
          total_profit=("Profit", "sum"),
          avg_shipping_days=("Shipping Days", "mean"),
          median_shipping_days=("Shipping Days", "median"),
      )
      .sort_values("avg_profit", ascending=False)
)

display(ship_mode_summary)

,orders,avg_profit,total_profit,avg_shipping_days,median_shipping_days
Ship Mode,,,,,
First Class,787,31.845643,48946.7535,2.182824,2.0
Second Class,964,29.449868,57191.6438,3.238414,3.0
Same Day,264,29.266591,15891.7589,0.044199,0.0
Standard Class,2994,27.495584,163983.6634,5.006875,5.0


Answer:
Compare both Avg Profit and Avg Shipping Days across modes.
If shipping days differ strongly but average profit is similar, Ship Mode mainly affects delivery speed rather than profitability.

## Q4 — Is there seasonality in Sales, by month or by day of week?

In [21]:
# Q4: seasonality in Sales -- by month and by weekday
monthly_sales = (
    df_prepared.groupby("Order Month", observed=True)["Sales"]
      .sum()
      .reindex(range(1, 13))
)

weekday_sales = (
    df_prepared.groupby("Order Weekday", observed=True)["Sales"]
      .sum()
      .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
)

print("Sales by month:")
display(monthly_sales.to_frame("Total Sales"))

print("\nSales by weekday:")
display(weekday_sales.to_frame("Total Sales"))

best_month = monthly_sales.idxmax()
worst_month = monthly_sales.idxmin()
best_day = weekday_sales.idxmax()
worst_day = weekday_sales.idxmin()

print(f"\nAnswer: Month {best_month} has the highest Sales (${monthly_sales.max():,.2f}); "
      f"Month {worst_month} has the lowest (${monthly_sales.min():,.2f}).")
print(f"By weekday, {best_day} is highest and {worst_day} is lowest.")



Sales by month:


,Total Sales
Order Month,
1,94924.8356
2,59751.2514
3,204959.8088
4,137188.7966
5,155028.8117
6,152718.6793
7,147148.0370
8,159044.0630
9,307600.8257



Sales by weekday:


,Total Sales
Order Weekday,
Monday,395924.1659
Tuesday,372918.7047
Wednesday,392859.0138
Thursday,298632.0632
Friday,206998.6180
Saturday,287363.7686
Sunday,340813.2381



Answer: Month 11 has the highest Sales ($351,916.69); Month 2 has the lowest ($59,751.25).
By weekday, Monday is highest and Friday is lowest.


## Q5 — Who are the most valuable customers, and how should we segment them using RFM?

We answer four business questions:

1. Which customer segments generate the highest revenue?
2. Which RFM segment generates the highest profit?
3. Which segments should receive retention campaigns?
4. Which segments should receive upselling/cross-selling offers?


In [34]:
top_customers = (
    rfm.sort_values(["Monetary"], ascending=False)
                [["Customer_ID", "Recency", "Frequency", "Monetary",
                  "RFM_Score", "RFM_Total", "Segment"]]
                .head(15)
)
print("Top 15 customers by RFM Monetary value:")
display(top_customers)

# Segment performance
rfm_segment_summary = (
    rfm.groupby("Segment", observed=True)
      .agg(
          customers=("Customer_ID", "nunique"),
          revenue=("Monetary", "sum"),
          avg_recency=("Recency", "mean"),
          avg_frequency=("Frequency", "mean"),
          avg_monetary=("Monetary", "mean"),
      )
      .sort_values("revenue", ascending=False)
)

rfm_segment_summary["revenue_share_%"] = (
    100 * rfm_segment_summary["revenue"] / rfm_segment_summary["revenue"].sum()
)

print("\nRFM segment performance:")
display(rfm_segment_summary)

top_revenue_segment = rfm_segment_summary["revenue"].idxmax()

print(f"Highest-revenue RFM segment: {top_revenue_segment}")

# Score-based targeting guidance, independent of the exact segment names.
segment_targeting = (
    rfm.groupby("Segment", observed=True)
      .agg(
          avg_R=("R_Score", "mean"),
          avg_F=("F_Score", "mean"),
          avg_M=("M_Score", "mean"),
          customers=("Customer_ID", "nunique"),
          revenue=("Monetary", "sum"),
      )
)

def targeting_action(row):
    # High R/F/M = valuable and active -> upsell/cross-sell
    if row["avg_R"] >= 3.5 and row["avg_F"] >= 3.5 and row["avg_M"] >= 3.5:
        return "Upsell / cross-sell"
    # Low recency with historically valuable customers -> retention/win-back
    if row["avg_R"] <= 2.5 and row["avg_M"] >= 3:
        return "Retention / win-back"
    # High frequency but not recently active -> retention
    if row["avg_F"] >= 3 and row["avg_R"] <= 2.5:
        return "Retention"
    return "Nurture / monitor"

segment_targeting["recommended_campaign"] = segment_targeting.apply(targeting_action, axis=1)

print("\nCampaign recommendation by RFM segment:")
display(segment_targeting.sort_values(["recommended_campaign", "revenue"], ascending=[True, False]))

print("\nAnswer:")
print(f"- Highest revenue segment: {top_revenue_segment}.")
print("- Retention campaigns should prioritize segments with low Recency scores but meaningful Frequency/Monetary value.")
print("- Upselling/cross-selling should prioritize segments with high Recency, Frequency, and Monetary scores.")


Top 15 customers by RFM Monetary value:


,Customer_ID,Recency,Frequency,Monetary,RFM_Score,RFM_Total,Segment
700,SM-20320,80,5,25043.050,325,10,At Risk (customers likely to churn)
741,TC-20980,400,5,19052.218,125,8,At Risk (customers likely to churn)
621,RB-19360,97,6,15117.339,335,11,At Risk (customers likely to churn)
730,TA-21385,70,4,14595.620,325,10,At Risk (customers likely to churn)
6,AB-10105,42,10,14473.571,455,14,At Risk (customers likely to churn)
434,KL-16645,48,12,14175.229,455,14,At Risk (customers likely to churn)
669,SC-20095,350,9,14142.334,155,11,At Risk (customers likely to churn)
327,HL-15040,44,6,12873.298,435,12,At Risk (customers likely to churn)
683,SE-20110,10,11,12209.438,555,15,At Risk (customers likely to churn)
131,CC-12370,44,5,12129.072,425,11,At Risk (customers likely to churn)



RFM segment performance:


,customers,revenue,avg_recency,avg_frequency,avg_monetary,revenue_share_%
Segment,,,,,,
Champions (best customers),296,956011.3908,71.925676,8.513514,3229.768212,41.647023
At Risk (customers likely to churn),70,643986.8886,119.342857,8.242857,9199.812694,28.054202
Lost / Low-Value (weak or lost customers),331,554459.4432,102.271903,4.703927,1675.104058,24.154090
Loyal / Potential (loyal & promising customers),96,141051.8497,559.218750,3.697917,1469.290101,6.144686


Highest-revenue RFM segment: Champions (best customers)

Campaign recommendation by RFM segment:


,avg_R,avg_F,avg_M,customers,revenue,recommended_campaign
Segment,,,,,,
At Risk (customers likely to churn),3.085714,3.971429,5.000000,70,643986.8886,Nurture / monitor
Lost / Low-Value (weak or lost customers),3.042296,2.048338,2.265861,331,554459.4432,Nurture / monitor
Loyal / Potential (loyal & promising customers),1.000000,1.572917,2.052083,96,141051.8497,Nurture / monitor
Champions (best customers),3.581081,4.297297,3.655405,296,956011.3908,Upsell / cross-sell



Answer:
- Highest revenue segment: Champions (best customers).
- Retention campaigns should prioritize segments with low Recency scores but meaningful Frequency/Monetary value.
- Upselling/cross-selling should prioritize segments with high Recency, Frequency, and Monetary scores.


## Q6 — Is Sales related to time?

We examine:
- Sales over Years
- Sales across Quarters
- Sales across Months

In [35]:
year_sales = (
    df_prepared.groupby("Order Year", observed=True)["Sales"]
      .sum()
      .sort_index()
)

quarter_sales = (
    df_prepared.groupby("Order Quarter", observed=True)["Sales"]
      .sum()
      .reindex([1, 2, 3, 4])
)

month_year_sales = (
    df_prepared.groupby(["Order Year", "Order Month"], observed=True)["Sales"]
      .sum()
      .reset_index()
)

weekday_weekend_sales = (
    df_prepared.assign(Day_Type=np.where(df_prepared["Is Weekend"], "Weekend", "Weekday"))
      .groupby("Day_Type", observed=True)["Sales"]
      .agg(["sum", "mean"])
      .rename(columns={"sum": "Total Sales", "mean": "Average Row Sales"})
)

print("Sales by Year:")
display(year_sales.to_frame("Total Sales"))

print("\nSales by Quarter:")
display(quarter_sales.to_frame("Total Sales"))

print("\nSales by Month:")
display(month_year_sales)

print("\nWeekday vs Weekend:")
display(weekday_weekend_sales)

if len(year_sales) >= 2:
    year_corr = pd.Series(year_sales.index.astype(float), index=year_sales.index).corr(year_sales)
    print(f"\nCorrelation between Year and annual Sales (descriptive): {year_corr:.3f}")

print("\nAnswer: Sales is related to time when the annual, quarterly, monthly, or weekday/weekend totals show systematic differences.")
print("Use the tables above to identify the strongest growth periods and recurring seasonal peaks.")


Sales by Year:


,Total Sales
Order Year,
2016,483966.1261
2017,470442.4490
2018,608532.4580
2019,732568.5392



Sales by Quarter:


,Total Sales
Order Quarter,
1,359635.8958
2,444936.2876
3,613792.9257
4,877144.4632



Sales by Month:


,Order Year,Order Month,Sales
0,2016,1,14236.8950
1,2016,2,4519.8920
2,2016,3,55691.0090
3,2016,4,28013.9730
4,2016,5,23648.2870
5,2016,6,34595.1276
6,2016,7,33946.3930
7,2016,8,27909.4685
8,2016,9,81777.3508
9,2016,10,31453.3930



Weekday vs Weekend:


,Total Sales,Average Row Sales
Day_Type,,
Weekday,1.667333e+06,229.850092
Weekend,6.281770e+05,229.933019



Correlation between Year and annual Sales (descriptive): 0.930

Answer: Sales is related to time when the annual, quarterly, monthly, or weekday/weekend totals show systematic differences.
Use the tables above to identify the strongest growth periods and recurring seasonal peaks.


## Q7 — Did Shipping Days affect any column?

We test:
- Number of Orders vs Shipping Days
- Shipping Days across Ship Modes
- Shipping Days across Regions
- Shipping Days vs Profit


In [36]:
orders_by_shipping_days = (
    df_prepared .groupby("Shipping Days", observed=True)
      .agg(
          number_of_orders=("Order ID", "nunique"),
          sales=("Sales", "sum"),
          profit=("Profit", "sum"),
          avg_profit=("Profit", "mean"),
      )
      .sort_index()
)

shipping_by_mode = (
    df_prepared.groupby("Ship Mode", observed=True)["Shipping Days"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("mean")
)

shipping_by_region = (
    df_prepared.groupby("Region", observed=True)["Shipping Days"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("mean")
)

shipping_profit_corr_pearson = df_prepared["Shipping Days"].corr(df_prepared["Profit"], method="pearson")
shipping_profit_corr_spearman = df_prepared["Shipping Days"].corr(df_prepared["Profit"], method="spearman")

print("Number of Orders and Shipping Days:")
display(orders_by_shipping_days)

print("\nShipping Days by Ship Mode:")
display(shipping_by_mode)

print("\nShipping Days by Region:")
display(shipping_by_region)

print(f"\nPearson correlation: Shipping Days vs Profit = {shipping_profit_corr_pearson:.3f}")
print(f"Spearman correlation: Shipping Days vs Profit = {shipping_profit_corr_spearman:.3f}")

print("\nAnswer:")
print("Shipping Days clearly varies by Ship Mode and can also differ by Region.")
print("The Profit correlations above quantify whether longer shipping is associated with higher/lower profitability.")


Number of Orders and Shipping Days:


,number_of_orders,sales,profit,avg_profit
Shipping Days,,,,
0,252,124907.6910,15385.9685,29.645411
1,182,67975.3312,7541.2269,20.436929
2,677,368358.0300,53089.9429,39.797558
3,508,203801.3748,26619.1305,26.565999
4,1401,631394.2453,71087.8694,25.672759
5,1085,494376.5397,58736.6093,27.067562
6,596,239979.4178,33220.6759,27.660846
7,308,164716.9425,20332.3962,32.688740



Shipping Days by Ship Mode:


,count,mean,median,min,max
Ship Mode,,,,,
Same Day,543,0.044199,0.0,0,1
First Class,1537,2.182824,2.0,1,5
Second Class,1942,3.238414,3.0,1,5
Standard Class,5964,5.006875,5.0,3,7



Shipping Days by Region:


,count,mean,median,min,max
Region,,,,,
East,2845,3.909315,4.0,0,7
West,3202,3.930668,4.0,0,7
South,1616,3.957921,4.0,0,7
Central,2323,4.057254,4.0,0,7



Pearson correlation: Shipping Days vs Profit = -0.005
Spearman correlation: Shipping Days vs Profit = -0.007

Answer:
Shipping Days clearly varies by Ship Mode and can also differ by Region.
The Profit correlations above quantify whether longer shipping is associated with higher/lower profitability.


## Q8 — How does Profit Margin vary across Categories?


In [37]:
category_margin = (
    df_prepared.groupby("Category", observed=True)
      .agg(
          sales=("Sales", "sum"),
          profit=("Profit", "sum"),
          quantity=("Quantity", "sum"),
          avg_row_margin=("Profit Margin", "mean"),
      )
)

category_margin["profit_margin"] = category_margin["profit"] / category_margin["sales"]
category_margin = category_margin.sort_values("profit_margin", ascending=False)

display(category_margin)

best_category_margin = category_margin["profit_margin"].idxmax()
print(f"\nAnswer: {best_category_margin} has the highest overall Profit Margin "
      f"({category_margin.loc[best_category_margin, 'profit_margin']:.2%}).")


,sales,profit,quantity,avg_row_margin,profit_margin
Category,,,,,
Technology,835759.7370,145386.1344,6926,15.613116,0.173957
Office Supplies,718317.7920,122247.4038,22891,13.784300,0.170186
Furniture,741432.0433,18380.2814,8023,3.870351,0.024790



Answer: Technology has the highest overall Profit Margin (17.40%).


## Q9 — What is the relationship between Profit Margin and Quantity?


In [39]:
margin_quantity_corr_pearson = df_prepared["Profit Margin"].corr(df_prepared["Quantity"], method="pearson")
margin_quantity_corr_spearman = df_prepared["Profit Margin"].corr(df_prepared["Quantity"], method="spearman")

margin_by_quantity = (
    df_prepared.groupby("Quantity", observed=True)
      .agg(
          avg_profit_margin=("Profit Margin", "mean"),
          total_sales=("Sales", "sum"),
          total_profit=("Profit", "sum"),
          rows=("Quantity", "size"),
      )
      .sort_index()
)

display(margin_by_quantity)

print(f"Pearson correlation (Profit Margin vs Quantity): {margin_quantity_corr_pearson:.3f}")
print(f"Spearman correlation (Profit Margin vs Quantity): {margin_quantity_corr_spearman:.3f}")
print("\nAnswer: Use the correlation and quantity-level table above to determine whether larger quantities are associated with higher or lower margins.")


,avg_profit_margin,total_sales,total_profit,rows
Quantity,,,,
1,12.279151,53251.9345,7440.4801,899
2,12.539646,288764.4278,38439.4548,2400
3,12.794962,421229.8407,56774.6046,2407
4,11.303226,323621.8744,44200.3044,1190
5,10.935577,415369.7365,49461.8520,1229
6,10.358644,207032.2614,10284.0402,571
7,11.477094,239908.3659,34286.9730,606
8,8.947001,117657.8400,10842.7248,256
9,12.931345,128505.5901,17687.8908,258


Pearson correlation (Profit Margin vs Quantity): -0.005
Spearman correlation (Profit Margin vs Quantity): -0.000

Answer: Use the correlation and quantity-level table above to determine whether larger quantities are associated with higher or lower margins.


## Q10 — How does Profit Margin vary across Customer Segments?


In [40]:
customer_segment_margin = (
    df_prepared.groupby("Segment", observed=True)
      .agg(
          sales=("Sales", "sum"),
          profit=("Profit", "sum"),
          quantity=("Quantity", "sum"),
          avg_row_margin=("Profit Margin", "mean"),
      )
)

customer_segment_margin["profit_margin"] = (
    customer_segment_margin["profit"] / customer_segment_margin["sales"]
)
customer_segment_margin = customer_segment_margin.sort_values("profit_margin", ascending=False)

display(customer_segment_margin)

best_segment_margin = customer_segment_margin["profit_margin"].idxmax()
print(f"\nAnswer: {best_segment_margin} has the highest overall Profit Margin "
      f"({customer_segment_margin.loc[best_segment_margin, 'profit_margin']:.2%}).")


,sales,profit,quantity,avg_row_margin,profit_margin
Segment,,,,,
Home Office,4.288950e+05,60170.4680,6725,14.240823,0.140292
Corporate,7.056020e+05,91821.2638,11605,12.114670,0.130132
Consumer,1.161013e+06,134022.0878,19510,11.201032,0.115436



Answer: Home Office has the highest overall Profit Margin (14.03%).


## Q11 — Which Products have the highest Profit per Unit?


In [45]:
product_features = (
    df_prepared.groupby(["Product ID", "Product Name", "Category", "Sub-Category"], observed=True)
      .agg(
          total_sales=("Sales", "sum"),
          total_profit=("Profit", "sum"),
          total_quantity=("Quantity", "sum"),
          avg_discount=("Discount", "mean"),
      )
      .reset_index()
)
product_features["Sales per Unit"] = (
    product_features["total_sales"] / product_features["total_quantity"]
)
product_features["Profit per Unit"] = (
    product_features["total_profit"] / product_features["total_quantity"]
)


In [46]:
# Use aggregate profit / aggregate quantity at product level.
# This avoids giving a tiny single-row transaction disproportionate influence.
top_products_profit_per_unit = (
    product_features.loc[product_features["total_quantity"] > 0]
                    .sort_values("Profit per Unit", ascending=False)
                    [["Product ID", "Product Name", "Category", "Sub-Category",
                      "total_quantity", "total_sales", "total_profit", "Profit per Unit"]]
                    .head(20)
)

display(top_products_profit_per_unit)

print("\nAnswer: The products at the top of this table have the highest aggregate Profit per Unit.")
print("Interpret the result together with total quantity/revenue so that a high ratio from very few units is not mistaken for a high-volume winner.")


,Product ID,Product Name,Category,Sub-Category,total_quantity,total_sales,total_profit,Profit per Unit
1640,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,20,61599.824,25199.9280,1259.996400
1674,TEC-MA-10002927,Canon imageCLASS MF7460 Monochrome Digital Las...,Technology,Machines,2,3991.980,1995.9900,997.995000
1643,TEC-MA-10000045,Zebra ZM400 Thermal Label Printer,Technology,Machines,6,6965.700,3343.5360,557.256000
1693,TEC-MA-10003979,Ativa V4110MDD Micro-Cut Shredder,Technology,Machines,11,7699.890,3772.9461,342.995100
1657,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,Technology,Machines,12,18374.895,4094.9766,341.248050
1656,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,11,14299.890,3717.9714,337.997400
1691,TEC-MA-10003673,Hewlett-Packard Desktjet 6988DT Refurbished Pr...,Technology,Machines,5,3404.500,1668.2050,333.641000
692,OFF-BI-10001120,Ibico EPK-21 Electric Binding System,Office Supplies,Binders,13,15875.916,3345.2823,257.329408
787,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,31,27453.384,7753.0390,250.098032
1636,TEC-CO-10003236,Canon Image Class D660 Copier,Technology,Copiers,7,3959.934,1691.9718,241.710257



Answer: The products at the top of this table have the highest aggregate Profit per Unit.
Interpret the result together with total quantity/revenue so that a high ratio from very few units is not mistaken for a high-volume winner.


## Step 3 — Overall Insight Synthesis

The tables above provide the evidence for all 14 questions. The main business themes to look for are:

- **Geography:** identify the strongest and weakest states by total Profit.
- **Discounting:** check whether higher Discount is associated with lower Profit.
- **Shipping:** Ship Mode strongly affects Shipping Days; compare whether that translates into meaningful Profit differences.
- **Time:** inspect annual growth and recurring monthly/weekday patterns.
- **Customers:** use RFM to distinguish high-value active customers from customers who are valuable but becoming inactive.
- **Margins:** compare Profit Margin across Category and Customer Segment rather than relying only on total Profit.
- **Unit economics:** Sales per Unit and Profit per Unit reveal product/customer differences that total Sales can hide.
- **Causality caveat:** correlations and group comparisons describe relationships in this dataset; they do not by themselves prove that one variable causes another.

### Existing cleaning caveat

Profit/Sales totals should remain consistent with the cleaning decision used to prepare the dataset: the legitimate `Order ID` + `Product ID` duplicate pairs were retained rather than collapsed.


## Step 4 — Handoff Prep for Visualization

All analysis tables are explicitly named so the next visualization/dashboard notebook can reuse them without recomputing the analysis.


In [49]:
handoff_manifest = {
    "df_clean": "row-level cleaned DataFrame",
    "df_prepared": "row-level cleaned + feature-engineered DataFrame",
    "customer_features": "one row per Customer ID with sales/profit/order metrics",
    "product_features": "one row per Product ID with sales/profit/quantity/unit metrics",
    "rfm": "one row per Customer ID with RFM scores and Segment",
    "state_profit": "total Profit per State",
    "discount_profit": "Profit by Discount",
    "ship_mode_summary": "profit and shipping time by Ship Mode",
    "monthly_sales": "total Sales by month",
    "weekday_sales": "total Sales by weekday",
    "year_sales": "total Sales by year",
    "quarter_sales": "total Sales by quarter",
    "rfm_segment_summary": "revenue/profit and RFM behavior by RFM Segment",
    "category_margin": "Profit Margin by Category",
    "segment_profit": "Profit by Customer Segment",
    "customer_segment_margin": "Profit Margin by Customer Segment",
    "segment_sales_per_unit": "Sales per Unit by Customer Segment",
    "top_products_profit_per_unit": "Products ranked by Profit per Unit",
}
for name, description in handoff_manifest.items():
    print(f"{name}: {description}")


df_clean: row-level cleaned DataFrame
df_prepared: row-level cleaned + feature-engineered DataFrame
customer_features: one row per Customer ID with sales/profit/order metrics
product_features: one row per Product ID with sales/profit/quantity/unit metrics
rfm: one row per Customer ID with RFM scores and Segment
state_profit: total Profit per State
discount_profit: Profit by Discount
ship_mode_summary: profit and shipping time by Ship Mode
monthly_sales: total Sales by month
weekday_sales: total Sales by weekday
year_sales: total Sales by year
quarter_sales: total Sales by quarter
rfm_segment_summary: revenue/profit and RFM behavior by RFM Segment
category_margin: Profit Margin by Category
segment_profit: Profit by Customer Segment
customer_segment_margin: Profit Margin by Customer Segment
segment_sales_per_unit: Sales per Unit by Customer Segment
top_products_profit_per_unit: Products ranked by Profit per Unit


## Step 5 — Persist Handoff Tables to Disk

Parquet is used for row/entity tables; CSV is used for compact analysis summaries.


In [52]:
import os

export_dir = "handoff_data"
os.makedirs(export_dir, exist_ok=True)

# Core/entity tables
df_prepared.to_parquet(f"{export_dir}/df_clean.parquet", engine="pyarrow")
customer_features.to_parquet(f"{export_dir}/customer_features.parquet", engine="pyarrow")
product_features.to_parquet(f"{export_dir}/product_features.parquet", engine="pyarrow")
rfm.to_parquet(f"{export_dir}/rfm.parquet", engine="pyarrow")

# Analysis summaries
summary_tables = {
    "state_profit": state_profit,
    "ship_mode_summary": ship_mode_summary,
    "monthly_sales": monthly_sales,
    "weekday_sales": weekday_sales,
    "year_sales": year_sales,
    "quarter_sales": quarter_sales,
    "rfm_segment_summary": rfm_segment_summary,
    "category_margin": category_margin,
    "customer_segment_margin": customer_segment_margin,
    "top_products_profit_per_unit": top_products_profit_per_unit,
    "orders_by_shipping_days": orders_by_shipping_days,
    "shipping_by_mode": shipping_by_mode,
    "shipping_by_region": shipping_by_region,
}

for name, table in summary_tables.items():
    table.to_csv(f"{export_dir}/{name}.csv")

print(f"Exported {len(os.listdir(export_dir))} files to '{export_dir}':")
for filename in sorted(os.listdir(export_dir)):
    print(" ", filename)


Exported 17 files to 'handoff_data':
  category_margin.csv
  customer_features.parquet
  customer_segment_margin.csv
  df_clean.parquet
  monthly_sales.csv
  orders_by_shipping_days.csv
  product_features.parquet
  quarter_sales.csv
  rfm.parquet
  rfm_segment_summary.csv
  ship_mode_summary.csv
  shipping_by_mode.csv
  shipping_by_region.csv
  state_profit.csv
  top_products_profit_per_unit.csv
  weekday_sales.csv
  year_sales.csv
